In [0]:
from pyspark.sql import functions as F


source_df = spark.read.option("header", True).option("inferSchema", True).csv("dbfs:/Volumes/lsg/pre_dm/tfsdl_lsg_ops_prod/inventory_rebalance/inputs/source_data.csv")
destination_df = spark.read.option("header", True).option("inferSchema", True).csv("dbfs:/Volumes/lsg/pre_dm/tfsdl_lsg_ops_prod/inventory_rebalance/inputs/destination_data.csv")
lanes_df = spark.read.option("header", True).option("inferSchema", True).csv("dbfs:/Volumes/lsg/pre_dm/tfsdl_lsg_ops_prod/inventory_rebalance/inputs/active_supply_lanes.csv")


def normalize_cols(df):
    return df.toDF(*[
        c.lower().replace(" ", "_") for c in df.columns
    ])

source_df = normalize_cols(source_df)
destination_df = normalize_cols(destination_df)
lanes_df = normalize_cols(lanes_df)


source_df = (
    source_df
    .withColumn("feasible_move_qty_row", F.least(
                                            F.col("Net_Available_After_Unallocated_SO"),
                                            F.col("max_available_to_move")
                                        ))
)

source_dir = "dbfs:/Volumes/lsg/pre_dm/tfsdl_lsg_ops_prod/inventory_rebalance/source_data"
source_df.write.mode("overwrite").format("parquet").save(source_dir)


lanes_df = (
    lanes_df
    .select(
        F.trim(F.col("from_branch")).alias("from_branch"),
        F.trim(F.col("to_branch")).alias("to_branch"),
        F.to_date(F.col("effective_from")).alias("effective_from"),
        F.to_date(F.col("effective_thru")).alias("effective_thru")

    )
    .dropna(subset=["from_branch", "to_branch", "effective_from"])
    .filter(
    (F.col("effective_from") <= F.current_date()) &
        (
            F.col("effective_thru").isNull() |
            (F.col("effective_thru") > F.current_date())
        )
    )
    .dropDuplicates()
)

In [0]:
display(spark.createDataFrame([(col,) for col in source_df.columns], ['column_name']))
display(spark.createDataFrame([(col,) for col in destination_df.columns], ['column_name']))
display(spark.createDataFrame([(col,) for col in lanes_df.columns], ['column_name']))

In [0]:
lane_candidates_df = (
    source_df.alias("s")
    .join(
        destination_df.alias("d"),
        (F.col("s.sku_number") == F.col("d.sku_number")) &
        (F.col("s.sku_source") == F.col("d.sku_source")) &
        (F.col("s.branch_code") != F.col("d.branch_code")),
        "inner"
    )
    .join(
        lanes_df.alias("l"),
        (F.col("s.branch_code") == F.col("l.from_branch")) &
        (F.col("d.branch_code") == F.col("l.to_branch")),
        "left"
    )
    .withColumn(
        "active_lane_flag",
        F.when(F.col("l.from_branch").isNotNull(), F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "destination_candidate_score_adjusted",
        F.col("d.candidate_score") + F.col("active_lane_flag")
    )
)


In [0]:
display(lane_candidates_df)

In [0]:
lane_candidates_model_df = lane_candidates_df.select(
    F.col("s.sku_number"),
    F.col("s.sku_source"),
    F.col("s.branch_code").alias("source_branch"),
    F.col("s.lot_number"),
    F.col("s.inventory_location_code"),
    F.col("s.unit_cost_pmar_amount").cast("double"),
    F.col("d.branch_code").alias("destination_branch"),
    F.col("s.Days_to_DNSA").alias("days_to_dnsa"),
    F.col("s.candidate_score").alias("source_candidate_score"),
    F.col("d.candidate_score").alias("destination_candidate_score"),
    F.col("active_lane_flag"),
    F.col("destination_candidate_score_adjusted"),
    F.col("s.feasible_move_qty_row"),
    F.col("s.max_available_to_move"),
    F.col("d.max_available_to_receive")
)





model_data_dir = "dbfs:/Volumes/lsg/pre_dm/tfsdl_lsg_ops_prod/inventory_rebalance/model_data"
lane_candidates_model_df.write.mode("overwrite").format("parquet").save(model_data_dir)




join_cols_source = ["sku_number", "sku_source"]
join_cols_dest = ["sku_number", "sku_source"]

source_cols = ["sku_number",
"branch_code",
"source_system_id",
"company_code",
"lead_time",
"stocking_type_label",
"safety_stock",
"unit_cost_pmar_amount",
"on_hand_quantity",
"available_quantity",
"lot_status_code",
"lot_number",
"inventory_location_code",
"dnsa_date",
"days_to_dnsa",
"compliance_soft_flag",
"lot_status_soft_flag",
"eo_reserve_soft_flag",
"intransit_soft_flag",
"allocated_type",
"total_open_so_quantity",
"net_available_after_unallocated_so",
"total_net_available_after_unallocated_so",
"reserve_years_window",
"usage_in_reserve_window",
"trailing_6m_company_usage",
"trailing_12m_company_usage",
"weeks_with_company_transaction",
"periods_with_company_transaction",
"weekly_usage",
"usage_protection_qty",
"weeks_of_supply",
"inv_vs_reserve_usage_window_flag",
"avail_inv_vs_ss_flag",
"safety_stock_quality_flag",
"trusted_safety_stock",
"weeks_of_supply_soft_flag",
"source_protected_quantity",
"candidate_score",
"max_available_to_move",
"feasible_move_qty_row"
]

dest_cols = ["branch_code",
"company_code",
"lead_time",
"stocking_type_label",
"safety_stock",
"spoke_flag",
"trailing_6m_company_usage",
"trailing_12m_company_usage",
"weeks_with_company_transaction",
"periods_with_company_transaction",
"reserve_years_window",
"usage_in_reserve_window",
"on_hand_quantity",
"inventory_value_usd_pmar_amount",
"open_order_quantity",
"available_quantity",
"weekly_usage",
"weeks_of_usage_target",
"safety_stock_quality_flag",
"trusted_safety_stock",
"weeks_of_supply",
"destination_target_qty",
"max_available_to_receive",
"weeks_with_transactions_soft_flag",
"periods_with_transactions_soft_flag",
"avail_inv_vs_ss_flag",
"candidate_score"

]

lane_cols = [
    F.col("active_lane_flag"),
    F.col("destination_candidate_score_adjusted")
]

lane_candidates_analysis_df = (
    lane_candidates_df
    .select(
        *[F.col(f"s.{col}").alias(f"source_{col}") for col in source_cols],
        *[F.col(f"d.{col}").alias(f"destination_{col}") for col in dest_cols],
        *lane_cols
    )
)

lane_candidates_analysis_dir = "dbfs:/Volumes/lsg/pre_dm/tfsdl_lsg_ops_prod/inventory_rebalance/lane_candidates_analysis_df"
lane_candidates_analysis_df.write.mode("overwrite").format("parquet").save(lane_candidates_analysis_dir)

In [0]:
display(spark.createDataFrame([(col,) for col in lane_candidates_analysis_df.columns], ['column_name']))